In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics  import  (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [2]:
phones_df = pd.read_csv('datasets/transformed_data/phones.csv')

X_data = phones_df.drop(columns=['price'])
y_data = phones_df['price']

X, X_test, y, y_test = train_test_split(
    X_data,
    y_data,
    test_size=.1
)

In [3]:
GEN_SIZE = 1
POP_SIZE = 15
MUTATION_PROB = 0.2
CROSSOVER_PROB = 0.8
SIGMA = 0.1
N_FEATURES = len(X_data.columns)

In [4]:
population = []

for _ in range(POP_SIZE):
    feature_mask = np.random.randint(
        0,
        2,
        N_FEATURES
    )
    
    if feature_mask.sum() == 0:
        feature_mask[np.random.randint(N_FEATURES)] = 1
    
    chromosome = np.concatenate([
        feature_mask,
        [
            np.random.randint(50, 501),
            np.random.randint(3, 31),
            np.random.randint(2, 16),
            np.random.randint(1, 11),
            np.random.randint(0, 3)
        ]]
    )
    
    population.append(chromosome)
    
population = np.array(population,dtype=object)

In [5]:

def decode(chromosome):
    
    feature_mask = chromosome[:N_FEATURES].astype(bool)
    
    n_estimators = int(chromosome[N_FEATURES])    
    max_depth = int(chromosome[N_FEATURES + 1])
    min_split = int(chromosome[N_FEATURES + 2])
    min_leaf = int(chromosome[N_FEATURES + 3])
    max_features_gene = int(chromosome[N_FEATURES + 4])
    
    max_features_map = {
        0:"sqrt",
        1:"log2",
        2: None
    }
    
    params = {
        "n_estimators" : n_estimators,
        "max_depth" : max_depth,
        "min_samples_split": min_split,
        "min_samples_leaf": min_leaf,
        "max_features": max_features_map[max_features_gene]
    }
    
    
    return feature_mask, params 

In [6]:

mask, params = decode(population[0])

mask, params

(array([False, False,  True, False,  True,  True,  True,  True, False,
        False,  True, False, False,  True,  True,  True, False,  True,
         True,  True,  True, False, False, False, False,  True, False,
        False,  True, False,  True, False,  True, False,  True,  True,
         True,  True,  True,  True, False,  True, False, False,  True,
        False, False, False,  True, False,  True]),
 {'n_estimators': 430,
  'max_depth': 22,
  'min_samples_split': 10,
  'min_samples_leaf': 8,
  'max_features': None})

In [7]:

from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestRegressor
global_best_score = 0
global_best_chromosome = []

kf = KFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

In [8]:

def fitness_function(chromosome, X, y):
    feature_mask, params = decode(chromosome)
    
    if(feature_mask.sum()) == 0:
        return
    
    X_selected = X.loc[:, feature_mask]
    
    model = RandomForestRegressor(
        **params,
        random_state=42
    )

    scores = cross_val_score(
        model,
        X_selected,
        y,
        cv=kf,
        scoring="r2"
    )
    print(scores)
    return np.mean(scores)

In [9]:
best_history = []
mean_history = []

In [10]:

def tournament_selection(population, fitness, k=3):
    selected = []
    
    pop_size = len(population)
    
    for _ in range(pop_size):
        
        idx = np.random.choice(
            pop_size,
            size=k,
            replace=False
        )
        
        tournament_fitness = fitness[idx]
        
        winner = idx[np.argmax(tournament_fitness)]
        
        selected.append(population[winner].copy())
        
    return np.array(selected, dtype=object)


def roulette_selection(population, fitness):
    
    adjusted = fitness - fitness.min() + 1e-10
    
    probability = adjusted / adjusted.sum()
    
    selected_index = np.random.choice(
        len(population),
        size = len(population),
        replace=True,
        p=probability
    )
    
    return np.array(
        [population[i].copy() for i in selected_index],
        dtype=object
    )
    
    
def boltzmann_selection(population, fitness, temperature):
    
    scaled = np.exp(
        (fitness-np.max(fitness)) / temperature
    )
    
    probability = scaled / scaled.sum()
    
    selected_index = np.random.choice(
        len(population),
        size=len(population),
        replace=True,
        p=probability
    )
    
    return np.array(
        [population[i].copy() for i in selected_index],
        dtype=object
    )

In [11]:
def crossover(parent1, parent2):
    child1 = parent1.copy()
    child2 = parent2.copy()
    
    for i in range(N_FEATURES):
        if np.random.rand() < 0.5:
            child1[i], child2[i] = child2[i], child1[i]
            
    for i in range(N_FEATURES, len(parent1)):
        if np.random.rand() < 0.5:
            child1[i], child2[i] = child2[i], child1[i]
            
    return child1, child2

In [12]:
def gaussian_mutation(value,  lower, upper):
    std = SIGMA * (upper - lower)
    value += np.random.normal(0, std)
    value = np.clip(value, lower, upper)
    
    return int(round(value))


def mutate(chromosome):
    chromosome = chromosome.copy()
    
    for i in range(N_FEATURES):
        if np.random.rand() < MUTATION_PROB:
            chromosome[i] = 1 - chromosome[i]
            
    if np.random.rand() < MUTATION_PROB:
        chromosome[N_FEATURES ] = gaussian_mutation(
            chromosome[N_FEATURES ], 50, 500
        )
    if np.random.rand() < MUTATION_PROB:
        chromosome[N_FEATURES + 1] = gaussian_mutation(
            chromosome[N_FEATURES + 1], 2, 50
        )
    if np.random.rand() < MUTATION_PROB:
        chromosome[N_FEATURES + 2] = gaussian_mutation(
            chromosome[N_FEATURES + 2], 2, 20
        )
    if np.random.rand() < MUTATION_PROB:
        chromosome[N_FEATURES + 3] = gaussian_mutation(
            chromosome[N_FEATURES + 3], 1, 20
        )
    if np.random.rand() < MUTATION_PROB:
        chromosome[N_FEATURES + 4] = np.random.randint(3)
        
    return chromosome

In [13]:
import time

start = time.perf_counter()

for generation in range(GEN_SIZE):
    print(f"Generation {generation}")
    fitness = np.array([
        fitness_function(chromosome, X, y )
        for chromosome in population
    ])
    best_history.append(fitness.max())
    mean_history.append(fitness.mean())

    best_idx = np.argmax(fitness)
    if fitness[best_idx] > global_best_score:
        global_best_score = fitness[best_idx]
        global_best = population[best_idx].copy()

    parents = tournament_selection(population, fitness, k=3)
    print(f"Fitness: {fitness.max():.4f}")
    offspring = []
    for j in range(0, POP_SIZE - 1, 2):

        parent1 = parents[j]
        parent2 = parents[j+1]
        
        if np.random.rand() < CROSSOVER_PROB:
            child1, child2 = crossover(parent1, parent2)
        else:
            child1 = parent1.copy()
            child2 = parent2.copy()
        
        child1 = mutate(child1)
        child2 = mutate(child2)
        
        offspring.extend([child1, child2])
    # offspring.append([child1, child2])
    population = np.array(offspring, dtype=object)

end = time.perf_counter()

Generation 0
[0.78809653 0.74923512 0.12976841 0.71368307 0.7838429  0.78582803
 0.7356452  0.71256961 0.70688663 0.72428131]
[0.76535053 0.76065381 0.12921391 0.69313261 0.78084428 0.80292172
 0.75048829 0.71539008 0.69868087 0.7113847 ]
[0.76792913 0.76225938 0.132306   0.68505863 0.79768798 0.78951822
 0.75944757 0.70774326 0.66342439 0.74440594]
[0.75810682 0.75479187 0.13023997 0.68483772 0.78349102 0.77976037
 0.75529892 0.70867525 0.67283409 0.71164611]
[0.74128619 0.76684233 0.12775538 0.70876212 0.79937918 0.84657468
 0.73990496 0.72977714 0.72305706 0.7737647 ]
[0.77564437 0.76920695 0.12746103 0.71872604 0.78812111 0.81218042
 0.74843604 0.70150669 0.71019921 0.7524503 ]
[0.79259439 0.7811043  0.13316597 0.72101544 0.79689822 0.82648926
 0.76539007 0.73567872 0.71398943 0.76064038]
[0.78849977 0.78008058 0.12890029 0.73741676 0.7864716  0.80147325
 0.76069913 0.72113014 0.71678515 0.74224052]
[0.80296785 0.79137564 0.1320051  0.75017234 0.79383307 0.84330599
 0.75711077 0.73

In [14]:
print(f"Waktu eksekusi pelatihan model: {(end - start):.2f}")

Waktu eksekusi pelatihan model: 268.48
